# BookBetter × RTHA — partnership KPIs in pandas

This notebook walks through the analysis behind the one-page report. The dataset is **synthetic** (see `src/generate_data.py`); the goal is to show how the metrics are built, not to claim real numbers.

Period: 1 July – 31 October 2026, the first four months after the RTHA booking engine went live.

In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.1f}'.format

bookings = pd.read_csv('../data/bookings.csv', parse_dates=['booking_date', 'departure_date'])
traffic = pd.read_csv('../data/engine_traffic.csv', parse_dates=['booking_date'])
bookings['month'] = bookings['booking_date'].dt.to_period('M')
bookings['with_saf'] = bookings['saf_coverage_pct'] > 0
print(bookings.shape, traffic.shape)
bookings.head()

(3753, 19) (123, 4)


,booking_id,booking_date,departure_date,destination,distance_km,trip_type,passengers,device,customer_id,gross_booking_value_eur,co2e_kg,saf_choice,saf_coverage_pct,saf_litres,saf_contribution_eur,co2e_avoided_kg,certificate_issued,month,with_saf
0,RTM-000001,2026-07-01,2026-07-21,Palma de Mallorca,1480,return,2,mobile,C00001,379.7,828.8,switched_off,0,0.0,0.0,0.0,False,2026-07,False
1,RTM-000002,2026-07-01,2026-07-12,Valencia,1530,one-way,2,desktop,C00002,286.3,428.4,kept_default,10,21.4,42.8,42.8,True,2026-07,True
2,RTM-000003,2026-07-01,2026-08-07,Nice,1020,one-way,2,mobile,C00003,195.4,285.6,switched_off,0,0.0,0.0,0.0,False,2026-07,False
3,RTM-000004,2026-07-01,2026-08-24,Vienna,960,one-way,2,mobile,C00004,174.8,268.8,kept_default,10,13.4,26.9,26.9,True,2026-07,True
4,RTM-000005,2026-07-01,2026-07-29,Alicante,1700,return,2,desktop,C00005,624.3,816.0,switched_off,0,0.0,0.0,0.0,False,2026-07,False


## 1. Headline numbers

The six figures an airport partner asks about first: volume, money, SAF uptake, SAF volume, CO₂e avoided, repeat travellers.

In [2]:
repeat = bookings.duplicated('customer_id', keep='first')
headline = pd.Series({
    'Bookings': len(bookings),
    'Passengers': bookings['passengers'].sum(),
    'Gross booking value (EUR)': bookings['gross_booking_value_eur'].sum(),
    'Partner revenue share @2.5% (EUR)': bookings['gross_booking_value_eur'].sum() * 0.025,
    'SAF uptake (% of bookings)': bookings['with_saf'].mean() * 100,
    'SAF funded (litres)': bookings['saf_litres'].sum(),
    'Traveller SAF contributions (EUR)': bookings['saf_contribution_eur'].sum(),
    'CO2e emitted (t)': bookings['co2e_kg'].sum() / 1000,
    'CO2e avoided (t)': bookings['co2e_avoided_kg'].sum() / 1000,
    'CO2e avoided (% of emitted)': bookings['co2e_avoided_kg'].sum() / bookings['co2e_kg'].sum() * 100,
    'Repeat bookings (%)': repeat.mean() * 100,
    'Certificates issued': bookings['certificate_issued'].sum(),
})
headline.to_frame('value')

,value
Bookings,"3,753.0"
Passengers,"7,601.0"
Gross booking value (EUR),"1,744,384.7"
Partner revenue share @2.5% (EUR),"43,609.6"
SAF uptake (% of bookings),75.0
SAF funded (litres),"107,733.2"
Traveller SAF contributions (EUR),"215,466.3"
CO2e emitted (t),"2,417.4"
CO2e avoided (t),215.5
CO2e avoided (% of emitted),8.9


## 2. Month by month

Does the default keep working after the launch effect fades? Uptake should hold or rise.

In [3]:
monthly = (bookings.groupby('month')
    .agg(bookings=('booking_id', 'size'),
         uptake_pct=('with_saf', lambda s: s.mean() * 100),
         saf_litres=('saf_litres', 'sum'),
         co2e_avoided_t=('co2e_avoided_kg', lambda s: s.sum() / 1000),
         gross_eur=('gross_booking_value_eur', 'sum')))
monthly

,bookings,uptake_pct,saf_litres,co2e_avoided_t,gross_eur
month,,,,,
2026-07,712,71.6,"18,592.7",37.2,"316,417.6"
2026-08,1154,71.8,"31,077.7",62.2,"579,171.3"
2026-09,869,76.9,"26,999.8",54.0,"397,401.4"
2026-10,1018,79.4,"31,062.9",62.1,"451,394.4"


## 3. What travellers do with the pre-selected 10 %

The single most important behavioural metric: the default is only valuable if people keep it.

In [4]:
choice = pd.crosstab(bookings['month'], bookings['saf_choice'], normalize='index') * 100
choice = choice[['increased', 'kept_default', 'reduced', 'switched_off']]
choice.round(1)

saf_choice,increased,kept_default,reduced,switched_off
month,,,,
2026-07,9.0,56.3,6.3,28.4
2026-08,8.1,56.8,6.8,28.2
2026-09,10.2,60.4,6.2,23.1
2026-10,10.2,64.7,4.4,20.6


In [5]:
on = bookings[bookings['with_saf']]
print(f"When SAF is on: avg coverage {on['saf_coverage_pct'].mean():.1f}% of emissions, "
      f"avg contribution EUR {on['saf_contribution_eur'].mean():.0f} per booking, "
      f"EUR {(on['saf_contribution_eur'] / on['passengers']).mean():.0f} per passenger")

When SAF is on: avg coverage 11.8% of emissions, avg contribution EUR 77 per booking, EUR 37 per passenger


## 4. Routes

Where the tonnes come from. Long leisure routes dominate; London City is busy but short.

In [6]:
routes = (bookings.groupby('destination')
    .agg(bookings=('booking_id', 'size'), passengers=('passengers', 'sum'),
         distance_km=('distance_km', 'first'),
         uptake_pct=('with_saf', lambda s: s.mean() * 100),
         co2e_avoided_t=('co2e_avoided_kg', lambda s: s.sum() / 1000))
    .sort_values('co2e_avoided_t', ascending=False))
routes

,bookings,passengers,distance_km,uptake_pct,co2e_avoided_t
destination,,,,,
Barcelona,535,1102,1240,75.5,29.8
Alicante,468,947,1700,73.9,28.1
Faro,312,655,1950,76.0,26.9
Malaga,402,779,1870,74.1,24.7
Lisbon,314,665,1880,74.2,21.5
Palma de Mallorca,267,528,1480,71.9,16.8
Porto,246,513,1620,76.8,15.5
Valencia,189,399,1530,76.7,14.1
Marrakech,141,266,2420,74.5,11.9


## 5. Funnel and devices

Session → search → booking → booking with SAF. Conversion is a BookBetter KPI; the SAF step is the partner's.

In [7]:
funnel = pd.Series({
    'Sessions': traffic['sessions'].sum(),
    'Searches': traffic['searches'].sum(),
    'Bookings': len(bookings),
    'Bookings with SAF': bookings['with_saf'].sum(),
}).to_frame('count')
funnel['step_conversion_pct'] = funnel['count'] / funnel['count'].shift(1) * 100
funnel

,count,step_conversion_pct
Sessions,179517,NaN
Searches,111083,61.9
Bookings,3753,3.4
Bookings with SAF,2815,75.0


In [8]:
bookings.groupby('device').agg(bookings=('booking_id', 'size'),
                               uptake_pct=('with_saf', lambda s: s.mean() * 100))

,bookings,uptake_pct
device,,
desktop,1575,74.8
mobile,2178,75.2


## 6. Assumptions used in the data

| Assumption | Value |
|---|---|
| CO₂e per passenger-km | 0.17 (< 800 km), 0.14 (800–1600 km), 0.12 (> 1600 km) kg |
| CO₂e avoided per litre of SAF | 2.0 kg (~80 % lifecycle reduction vs fossil jet) |
| SAF price gap paid by traveller | €2.00 per litre |
| Default contribution | 10 % of the booking's emissions, adjustable |
| Partner revenue share | 2.5 % of gross booking value (illustrative) |

The point of the exercise is the metric choice and the story, not the exact constants — swap them in `src/generate_data.py` and rebuild.